In [27]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import pandas as pd

options = Options()
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
options.add_argument(f"user-agent={user_agent}")
options.add_argument("--headless")

driver = webdriver.Chrome(options=options)

base_url = "https://divar.ir/s/tehran/mobile-phones?q= گوشی%20a55"
all_listings = []

for page in range(1, 10):  # بررسی ۹ صفحه اول
    print(f"در حال دریافت صفحه {page}...")
    url = f"{base_url}&page={page}"
    driver.get(url)
    time.sleep(5)

    ads = driver.find_elements(By.CSS_SELECTOR, "article.unsafe-kt-post-card")
    if not ads:
        print("آگهی بیشتری یافت نشد.")
        break

    for ad in ads:
        try:
            title = ad.find_element(By.CSS_SELECTOR, "h2.unsafe-kt-post-card__title").text
            
            # استفاده از سلکتور جدید برای قیمت
            price_elem = ad.find_element(By.CSS_SELECTOR, 
                                          "div.unsafe-kt-post-card__info > div:nth-child(3)").text
            
            # بعضی وقتها قیمت توافقی هست یا خالی میاد
            if not price_elem or "توافقی" in price_elem:
                price_elem = "قیمت توافقی"
            
            try:
                location = ad.find_element(By.CSS_SELECTOR,
                                           "span.unsafe-kt-post-card__bottom-description").get_attribute("title")
            except:
                location = "نامشخص"

            link = ad.find_element(By.CSS_SELECTOR, "a.unsafe-kt-post-card__action").get_attribute("href")
            if not link.startswith("http"):
                link = "https://divar.ir " + link.strip()  # رفع فاصله‌های اضافی

            listing = {
                "title": title,
                "location": location,
                "price": price_elem,
                "link": link
            }
            all_listings.append(listing)
        except Exception as e:
            print("خطا در خواندن یک آگهی:", e)

driver.quit()

# ذخیره در فایل CSV
df = pd.DataFrame(all_listings)
df.to_csv("samsung_a55_listings.csv", index=False)
print(f"\n✅ تعداد کل آگهی‌های جمع‌آوری شده: {len(df)}")

در حال دریافت صفحه 1...
در حال دریافت صفحه 2...
در حال دریافت صفحه 3...
در حال دریافت صفحه 4...
در حال دریافت صفحه 5...
در حال دریافت صفحه 6...
در حال دریافت صفحه 7...
در حال دریافت صفحه 8...
در حال دریافت صفحه 9...

✅ تعداد کل آگهی‌های جمع‌آوری شده: 126


In [ ]:
import pandas as pd

df = pd.read_csv("samsung_a55_listings.csv")

# برای نمایش کل دیتا به سبک head (اما کامل)
df  # کافی است فقط همین را بنویسی


In [29]:
print("ستون‌های موجود در فایل:")
print(df.columns)

ستون‌های موجود در فایل:
Index(['title', 'location', 'price', 'link'], dtype='object')


In [32]:
import pandas as pd

# خواندن CSV ذخیره‌شده
df = pd.read_csv("samsung_a55_listings.csv")

# پاک‌سازی و تبدیل قیمت به عدد (برای مقایسه)
def extract_price(price_str):
    try:
        # حذف "تومان" و جداکننده‌های هزارگان
        price_num = price_str.replace("تومان", "").replace(",", "").strip()
        return int(price_num)
    except:
        return None

df["numeric_price"] = df["price"].apply(extract_price)


In [33]:
# فیلترهایی که حداقل یکی باید برقرار باشد
condition_ram = df["title"].str.contains("رم ?8|8 ?گیگ", case=False, regex=True, na=False)
condition_memory = df["title"].str.contains("256", case=False, regex=True, na=False)
condition_price = df["numeric_price"] > 20_000_000

# نهایی: فقط ردیف‌هایی که یکی از این سه شرط را دارند
filtered_df = df[condition_ram | condition_memory | condition_price]


In [38]:
filtered_df[["title", "location", "price"]]


,title,location,price
1,a55.256gig 5G,دیروز در تهرانپارس شرقی,"۲۳,۹۰۰,۰۰۰ تومان"
6,سامسونگ Galaxy A55 با حافظهٔ ۲۵۶ گیگابایت,پریروز در نعمت‌آباد,"۲۶,۰۰۰,۰۰۰ تومان"
7,a55 ۱۲۸,پریروز در آذری,"۲۴,۰۰۰,۰۰۰ تومان"
8,سامسونگ Galaxy A55 با حافظهٔ ۲۵۶ گیگابایت,پریروز در آبشار تهران,"۲۵,۰۰۰,۰۰۰ تومان"
9,سامسونگ a55 با حافظهٔ ۲۵۶ گیگ و رام ۱۲ گیگ,پریروز در یاخچی‌آباد,"۲۸,۵۰۰,۰۰۰ تومان"
10,سامسونگ Galaxy A55 با حافظهٔ ۲۵۶ گیگ ویتنام آکبند,پریروز در افسریه,"۲۸,۰۰۰,۰۰۰ تومان"
11,سامسونگ Galaxy A55 با حافظهٔ ۲۵۶ گیگابایت,پریروز در تهرانپارس غربی,"۱۴۰,۰۰۰,۰۰۰ تومان"
13,A55حافظه 256,پریروز در آهنگ,"۲۴,۰۰۰,۰۰۰ تومان"
14,گوشی موبایل سامسونگ گلکسی a55,۳ روز پیش در اوقاف,"۳۵,۰۰۰,۰۰۰ تومان"
17,سامسونگ a55 رم ۸ حافظه ۲۵۶ رنگ سرمه ای,در تهران‌نو,"۲۵,۰۰۰,۰۰۰ تومان"
